# morphological_quantification_2026-01-02 — 05g_transverse_distance_profiles_merged

**Feeds:** Fig 3f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 05g | Transverse Distance Profiles (Merged Cohorts)

Merge transverse-distance profiles from the 2026-01-02 and baseline morphological quantification
cohorts, then replot the combined distributions with shared distance bins.


## Cell Guide

- `Setup`: resolve the project root, import helper code, and define bins.
- `Load Inputs`: read per-file transverse profiles from both cohorts.
- `Interpolate To Shared Bins`: rebin both cohorts to a common transverse distance grid.
- `Summaries`: compute mean/SD and n for each metric across all organoids.
- `Plots`: generate the same transverse-distance plots from the merged data only.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif cwd.name == "executed_notebooks" and cwd.parent.name == "results":
    ROOT = cwd.parent.parent
elif (cwd / "notebooks").exists():
    ROOT = cwd
elif cwd.parent.name == "results" and (cwd.parent.parent / "notebooks").exists():
    ROOT = cwd.parent.parent
else:
    raise RuntimeError(
        "Run this notebook from the project root, notebooks/, or results/executed_notebooks/."
    )

SCRIPTS_DIR = ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import morphology_quantification_helpers as mqh

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 120)


In [ ]:
BASELINE_ROOT = Path("<analysis-root>/morphological_quantification")
MERGED_ROOT = ROOT

BASELINE_TABLE = BASELINE_ROOT / "results" / "tables" / "05g_transverse_distance_profiles_by_file.tsv"
COHORT_TABLE = MERGED_ROOT / "results" / "tables" / "05g_transverse_distance_profiles_by_file.tsv"

OUTPUT_TABLE_PATH = MERGED_ROOT / "results" / "tables" / "05g_transverse_distance_profiles_merged_by_file.tsv"
SUMMARY_TABLE_PATH = MERGED_ROOT / "results" / "tables" / "05g_transverse_distance_profiles_merged_summary.tsv"
OUTPUT_QC_DIR = MERGED_ROOT / "results" / "qc" / "transverse_distance_profiles_merged"
OUTPUT_QC_DIR.mkdir(parents=True, exist_ok=True)

PLOT_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles.png"
PLOT_PATH_CI = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_ci.png"
PLOT_FRACTION_NORM_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm.png"
PLOT_FRACTION_NORM_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_ci.png"
PLOT_FRACTION_MINMAX_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax.png"
PLOT_FRACTION_MINMAX_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_ci.png"
PLOT_FRACTION_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only.png"
PLOT_FRACTION_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only_ci.png"
PLOT_FRACTION_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only.png"
PLOT_FRACTION_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only_ci.png"
PLOT_FRACTION_MINMAX_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only.png"
PLOT_FRACTION_MINMAX_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only_ci.png"
PLOT_INTENSITY_RAW_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw.png"
PLOT_INTENSITY_RAW_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_ci.png"
PLOT_INTENSITY_NORM_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm.png"
PLOT_INTENSITY_NORM_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_ci.png"
PLOT_INTENSITY_ALL_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all.png"
PLOT_INTENSITY_ALL_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_ci.png"
PLOT_INTENSITY_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only.png"
PLOT_INTENSITY_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only_ci.png"
PLOT_INTENSITY_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only.png"
PLOT_INTENSITY_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only_ci.png"
PLOT_INTENSITY_ALL_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only.png"
PLOT_INTENSITY_ALL_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only_ci.png"
PLOT_SOFT_RAW_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw.png"
PLOT_SOFT_RAW_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_ci.png"
PLOT_SOFT_NORM_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm.png"
PLOT_SOFT_NORM_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_ci.png"
PLOT_SOFT_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only.png"
PLOT_SOFT_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only_ci.png"
PLOT_SOFT_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only.png"
PLOT_SOFT_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only_ci.png"

MARKERS = ["mesp2", "foxf1", "pax8"]
COLOR_MAP = {"mesp2": "#ef4444", "pax8": "#facc15", "foxf1": "#22d3ee"}
N_BINS = 24


## Load Inputs


In [ ]:
if not BASELINE_TABLE.exists():
    raise RuntimeError(f"Missing baseline table: {BASELINE_TABLE}")
if not COHORT_TABLE.exists():
    raise RuntimeError(f"Missing cohort table: {COHORT_TABLE}")

base_df = pd.read_csv(BASELINE_TABLE, sep="\t")
cohort_df = pd.read_csv(COHORT_TABLE, sep="\t")
base_df["cohort_label"] = "baseline"
cohort_df["cohort_label"] = "2026-01-02"
merged_df = pd.concat([base_df, cohort_df], ignore_index=True)


## Interpolate Onto Shared Bins


In [ ]:
global_max_um = float(np.nanmax(merged_df["bin_center_um"].to_numpy(dtype=float)))
if not np.isfinite(global_max_um) or global_max_um <= 0:
    raise RuntimeError("Invalid transverse distance range.")

bin_edges = np.linspace(0.0, global_max_um, int(N_BINS) + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

def interp_to_bins(x, y, new_x):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if np.sum(mask) < 2:
        return np.full_like(new_x, np.nan, dtype=float)
    x = x[mask]
    y = y[mask]
    order = np.argsort(x)
    x = x[order]
    y = y[order]
    interp_vals = np.interp(new_x, x, y)
    interp_vals[(new_x < x[0]) | (new_x > x[-1])] = np.nan
    return interp_vals

metrics = [
    "positive_fraction",
    "positive_intensity_fraction",
    "intensity_fraction",
    "soft_intensity_fraction",
]

rows = []
grouped = merged_df.groupby(["cohort_label", "file_path", "marker_key"], sort=True)
for (cohort_label, file_path, marker_key), group in grouped:
    file_id = int(group["file_id"].iloc[0])
    x = group["bin_center_um"].to_numpy(dtype=float)
    for metric in metrics:
        y = group[metric].to_numpy(dtype=float)
        interp_vals = interp_to_bins(x, y, bin_centers)
        for idx, center_um in enumerate(bin_centers):
            rows.append(
                {
                    "cohort_label": cohort_label,
                    "file_id": file_id,
                    "file_path": file_path,
                    "marker_key": marker_key,
                    "bin_index": int(idx),
                    "bin_center_um": float(center_um),
                    metric: float(interp_vals[idx]) if np.isfinite(interp_vals[idx]) else np.nan,
                }
            )

interp_df = pd.DataFrame(rows)
interp_df = interp_df.groupby(
    ["cohort_label", "file_id", "file_path", "marker_key", "bin_index", "bin_center_um"],
    as_index=False,
).first()

interp_df.to_csv(OUTPUT_TABLE_PATH, sep="\t", index=False)


## Summaries


In [ ]:
summary_rows = []
for marker_key in MARKERS:
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key].copy()
    for bin_idx, bin_center_um in enumerate(bin_centers):
        bin_sub = sub.loc[sub["bin_index"].astype(int) == int(bin_idx)]
        values = pd.to_numeric(bin_sub["positive_fraction"], errors="coerce")
        values_intensity = pd.to_numeric(bin_sub["positive_intensity_fraction"], errors="coerce")
        values_intensity_all = pd.to_numeric(bin_sub["intensity_fraction"], errors="coerce")
        values_soft = pd.to_numeric(bin_sub["soft_intensity_fraction"], errors="coerce")
        mean_val = float(np.nanmean(values)) if values.notna().any() else np.nan
        std_val = float(np.nanstd(values)) if values.notna().any() else np.nan
        mean_intensity = float(np.nanmean(values_intensity)) if values_intensity.notna().any() else np.nan
        std_intensity = float(np.nanstd(values_intensity)) if values_intensity.notna().any() else np.nan
        mean_intensity_all = (
            float(np.nanmean(values_intensity_all)) if values_intensity_all.notna().any() else np.nan
        )
        std_intensity_all = (
            float(np.nanstd(values_intensity_all)) if values_intensity_all.notna().any() else np.nan
        )
        mean_soft = float(np.nanmean(values_soft)) if values_soft.notna().any() else np.nan
        std_soft = float(np.nanstd(values_soft)) if values_soft.notna().any() else np.nan
        summary_rows.append(
            {
                "marker_key": marker_key,
                "bin_index": int(bin_idx),
                "bin_center_um": float(bin_center_um),
                "mean_fraction": mean_val,
                "std_fraction": std_val,
                "mean_intensity_fraction": mean_intensity,
                "std_intensity_fraction": std_intensity,
                "mean_intensity_all_fraction": mean_intensity_all,
                "std_intensity_all_fraction": std_intensity_all,
                "mean_soft_intensity_fraction": mean_soft,
                "std_soft_intensity_fraction": std_soft,
                "n_organoids": int(values.notna().sum()),
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_TABLE_PATH, sep="\t", index=False)


## Plot (Mean-Only Variants)


In [ ]:
def plot_mean_only(metric, std_col, title, ylabel, out_path, scale="none", ci=False):
    fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
    max_mean_local = 0.0

    for marker_key in MARKERS:
        color = COLOR_MAP.get(marker_key, "0.4")
        sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
        x = sub["bin_center_um"].to_numpy(dtype=float)
        mean = sub[metric].to_numpy(dtype=float)
        std = sub[std_col].to_numpy(dtype=float)
        n = sub["n_organoids"].to_numpy(dtype=float)

        if scale == "peak":
            scale_val = float(np.nanmax(mean))
            if np.isfinite(scale_val) and scale_val > 0:
                mean = mean / scale_val
                std = std / scale_val
        elif scale == "minmax":
            min_val = float(np.nanmin(mean))
            max_val = float(np.nanmax(mean))
            denom = max_val - min_val
            if np.isfinite(denom) and denom > 0:
                mean = (mean - min_val) / denom
                std = std / denom

        band = 1.96 * std / np.sqrt(n) if ci else std
        ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
        ax.fill_between(x, mean - band, mean + band, color=color, alpha=0.18, linewidth=0)
        max_mean_local = max(max_mean_local, float(np.nanmax(mean)))

    ax.set_xlabel("Absolute transverse distance from axis (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25, linewidth=0.6)
    ax.legend(frameon=False)

    if np.isfinite(max_mean_local) and max_mean_local > 0:
        ymin, ymax = ax.get_ylim()
        upper = max(ymax, max_mean_local)
        upper = min(upper, 2 * max_mean_local)
        ax.set_ylim(0, upper)

    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".svg"), bbox_inches="tight")
    plt.close(fig)

plot_mean_only(
    metric="mean_fraction",
    std_col="std_fraction",
    title="Marker positivity vs transverse distance (merged cohorts, mean only)",
    ylabel="Fraction of pixels that are marker-positive",
    out_path=PLOT_FRACTION_MEAN_PATH,
    scale="none",
    ci=False,
)
plot_mean_only(
    metric="mean_fraction",
    std_col="std_fraction",
    title="Marker positivity vs transverse distance (merged, mean only, 95% CI)",
    ylabel="Fraction of pixels that are marker-positive",
    out_path=PLOT_FRACTION_MEAN_CI_PATH,
    scale="none",
    ci=True,
)
plot_mean_only(
    metric="mean_fraction",
    std_col="std_fraction",
    title="Marker positivity vs transverse distance (merged, normalized, mean only)",
    ylabel="Positive fraction (mean peak normalized to 1)",
    out_path=PLOT_FRACTION_NORM_MEAN_PATH,
    scale="peak",
    ci=False,
)
plot_mean_only(
    metric="mean_fraction",
    std_col="std_fraction",
    title="Marker positivity vs transverse distance (merged, normalized, mean only, 95% CI)",
    ylabel="Positive fraction (mean peak normalized to 1)",
    out_path=PLOT_FRACTION_NORM_MEAN_CI_PATH,
    scale="peak",
    ci=True,
)
plot_mean_only(
    metric="mean_fraction",
    std_col="std_fraction",
    title="Marker positivity vs transverse distance (merged, min-max, mean only)",
    ylabel="Positive fraction (mean trace min-max to [0,1])",
    out_path=PLOT_FRACTION_MINMAX_MEAN_PATH,
    scale="minmax",
    ci=False,
)
plot_mean_only(
    metric="mean_fraction",
    std_col="std_fraction",
    title="Marker positivity vs transverse distance (merged, min-max, mean only, 95% CI)",
    ylabel="Positive fraction (mean trace min-max to [0,1])",
    out_path=PLOT_FRACTION_MINMAX_MEAN_CI_PATH,
    scale="minmax",
    ci=True,
)
plot_mean_only(
    metric="mean_intensity_fraction",
    std_col="std_intensity_fraction",
    title="Marker-positive intensity vs transverse distance (merged, mean only)",
    ylabel="Fraction of positive intensity in bin",
    out_path=PLOT_INTENSITY_RAW_MEAN_PATH,
    scale="none",
    ci=False,
)
plot_mean_only(
    metric="mean_intensity_fraction",
    std_col="std_intensity_fraction",
    title="Marker-positive intensity vs transverse distance (merged, mean only, 95% CI)",
    ylabel="Fraction of positive intensity in bin",
    out_path=PLOT_INTENSITY_RAW_MEAN_CI_PATH,
    scale="none",
    ci=True,
)
plot_mean_only(
    metric="mean_intensity_fraction",
    std_col="std_intensity_fraction",
    title="Marker-positive intensity vs transverse distance (merged, normalized, mean only)",
    ylabel="Positive intensity (mean peak normalized to 1)",
    out_path=PLOT_INTENSITY_NORM_MEAN_PATH,
    scale="peak",
    ci=False,
)
plot_mean_only(
    metric="mean_intensity_fraction",
    std_col="std_intensity_fraction",
    title="Marker-positive intensity vs transverse distance (merged, normalized, mean only, 95% CI)",
    ylabel="Positive intensity (mean peak normalized to 1)",
    out_path=PLOT_INTENSITY_NORM_MEAN_CI_PATH,
    scale="peak",
    ci=True,
)
plot_mean_only(
    metric="mean_intensity_all_fraction",
    std_col="std_intensity_all_fraction",
    title="Marker intensity vs transverse distance (merged, all mask pixels, mean only)",
    ylabel="Fraction of all corrected intensity in bin",
    out_path=PLOT_INTENSITY_ALL_MEAN_PATH,
    scale="none",
    ci=False,
)
plot_mean_only(
    metric="mean_intensity_all_fraction",
    std_col="std_intensity_all_fraction",
    title="Marker intensity vs transverse distance (merged, all mask pixels, mean only, 95% CI)",
    ylabel="Fraction of all corrected intensity in bin",
    out_path=PLOT_INTENSITY_ALL_MEAN_CI_PATH,
    scale="none",
    ci=True,
)
plot_mean_only(
    metric="mean_soft_intensity_fraction",
    std_col="std_soft_intensity_fraction",
    title="Marker intensity vs transverse distance (merged, soft threshold, mean only)",
    ylabel="Fraction of soft-threshold intensity in bin",
    out_path=PLOT_SOFT_RAW_MEAN_PATH,
    scale="none",
    ci=False,
)
plot_mean_only(
    metric="mean_soft_intensity_fraction",
    std_col="std_soft_intensity_fraction",
    title="Marker intensity vs transverse distance (merged, soft threshold, mean only, 95% CI)",
    ylabel="Fraction of soft-threshold intensity in bin",
    out_path=PLOT_SOFT_RAW_MEAN_CI_PATH,
    scale="none",
    ci=True,
)
plot_mean_only(
    metric="mean_soft_intensity_fraction",
    std_col="std_soft_intensity_fraction",
    title="Marker intensity vs transverse distance (merged, soft threshold, normalized, mean only)",
    ylabel="Soft-threshold intensity (mean peak normalized to 1)",
    out_path=PLOT_SOFT_NORM_MEAN_PATH,
    scale="peak",
    ci=False,
)
plot_mean_only(
    metric="mean_soft_intensity_fraction",
    std_col="std_soft_intensity_fraction",
    title="Marker intensity vs transverse distance (merged, soft threshold, normalized, mean only, 95% CI)",
    ylabel="Soft-threshold intensity (mean peak normalized to 1)",
    out_path=PLOT_SOFT_NORM_MEAN_CI_PATH,
    scale="peak",
    ci=True,
)


## Plot (Positive Fraction)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["positive_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of pixels that are marker-positive")
ax.set_title("Marker positivity vs transverse distance (merged cohorts)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
if np.isfinite(max_mean) and max_mean > 0:
    ymin, ymax = ax.get_ylim()
    upper = max(ymax, max_mean)
    upper = min(upper, 2 * max_mean)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote profile plot:", PLOT_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["positive_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of pixels that are marker-positive")
ax.set_title("Marker positivity vs transverse distance (merged, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
if np.isfinite(max_mean) and max_mean > 0:
    ymin, ymax = ax.get_ylim()
    upper = max(ymax, max_mean)
    upper = min(upper, 2 * max_mean)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_PATH_CI, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_PATH_CI.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote profile plot (CI):", PLOT_PATH_CI.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, Mean Peak Normalized)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive fraction (mean peak normalized to 1)")
ax.set_title("Marker positivity vs transverse distance (merged, normalized)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
ymin, ymax = ax.get_ylim()
max_mean_norm = float(np.nanmax(mean)) if np.isfinite(np.nanmax(mean)) else max_mean
if np.isfinite(max_mean_norm) and max_mean_norm > 0:
    upper = max(ymax, max_mean_norm)
    upper = min(upper, 2 * max_mean_norm)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_FRACTION_NORM_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_FRACTION_NORM_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized fraction plot:", PLOT_FRACTION_NORM_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, Mean Peak Normalized, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive fraction (mean peak normalized to 1)")
ax.set_title("Marker positivity vs transverse distance (merged, normalized, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
ymin, ymax = ax.get_ylim()
max_mean_norm = float(np.nanmax(mean)) if np.isfinite(np.nanmax(mean)) else max_mean
if np.isfinite(max_mean_norm) and max_mean_norm > 0:
    upper = max(ymax, max_mean_norm)
    upper = min(upper, 2 * max_mean_norm)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_FRACTION_NORM_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_FRACTION_NORM_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized fraction plot (CI):", PLOT_FRACTION_NORM_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, Mean Trace Min-Max)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0

minmax_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    mean_vals = summary_sub["mean_fraction"].to_numpy(dtype=float)
    min_val = float(np.nanmin(mean_vals))
    max_val = float(np.nanmax(mean_vals))
    minmax_by_marker[marker_key] = (min_val, max_val)

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    min_val, max_val = minmax_by_marker.get(marker_key, (np.nan, np.nan))
    denom = max_val - min_val if np.isfinite(max_val) and np.isfinite(min_val) else np.nan
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_fraction"].to_numpy(dtype=float)
        if np.isfinite(denom) and denom > 0:
            y = (y - min_val) / denom
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    if np.isfinite(denom) and denom > 0:
        mean = (mean - min_val) / denom
        std = std / denom
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive fraction (mean trace min-max to [0,1])")
ax.set_title("Marker positivity vs transverse distance (merged, min-max)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
ymin, ymax = ax.get_ylim()
max_mean_norm = float(np.nanmax(mean)) if np.isfinite(np.nanmax(mean)) else max_mean
if np.isfinite(max_mean_norm) and max_mean_norm > 0:
    upper = max(ymax, max_mean_norm)
    upper = min(upper, 2 * max_mean_norm)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_FRACTION_MINMAX_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_FRACTION_MINMAX_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote min-max fraction plot:", PLOT_FRACTION_MINMAX_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, Mean Trace Min-Max, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0

minmax_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    mean_vals = summary_sub["mean_fraction"].to_numpy(dtype=float)
    min_val = float(np.nanmin(mean_vals))
    max_val = float(np.nanmax(mean_vals))
    minmax_by_marker[marker_key] = (min_val, max_val)

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    min_val, max_val = minmax_by_marker.get(marker_key, (np.nan, np.nan))
    denom = max_val - min_val if np.isfinite(max_val) and np.isfinite(min_val) else np.nan
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_fraction"].to_numpy(dtype=float)
        if np.isfinite(denom) and denom > 0:
            y = (y - min_val) / denom
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    if np.isfinite(denom) and denom > 0:
        mean = (mean - min_val) / denom
        std = std / denom
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive fraction (mean trace min-max to [0,1])")
ax.set_title("Marker positivity vs transverse distance (merged, min-max, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
ymin, ymax = ax.get_ylim()
max_mean_norm = float(np.nanmax(mean)) if np.isfinite(np.nanmax(mean)) else max_mean
if np.isfinite(max_mean_norm) and max_mean_norm > 0:
    upper = max(ymax, max_mean_norm)
    upper = min(upper, 2 * max_mean_norm)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_FRACTION_MINMAX_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_FRACTION_MINMAX_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote min-max fraction plot (CI):", PLOT_FRACTION_MINMAX_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Intensity, Raw)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_intensity_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["positive_intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_fraction"].to_numpy(dtype=float)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of positive intensity in bin")
ax.set_title("Marker-positive intensity vs transverse distance (merged)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
if np.isfinite(max_mean) and max_mean > 0:
    ymin, ymax = ax.get_ylim()
    upper = max(ymax, max_mean)
    upper = min(upper, 2 * max_mean)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_INTENSITY_RAW_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_RAW_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote intensity profile plot:", PLOT_INTENSITY_RAW_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Intensity, Raw, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_intensity_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["positive_intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of positive intensity in bin")
ax.set_title("Marker-positive intensity vs transverse distance (merged, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
if np.isfinite(max_mean) and max_mean > 0:
    ymin, ymax = ax.get_ylim()
    upper = max(ymax, max_mean)
    upper = min(upper, 2 * max_mean)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_INTENSITY_RAW_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_RAW_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote intensity profile plot (CI):", PLOT_INTENSITY_RAW_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Intensity, Mean Peak Normalized)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_intensity_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_fraction"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive intensity (mean peak normalized to 1)")
ax.set_title("Marker-positive intensity vs transverse distance (merged, normalized)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
ymin, ymax = ax.get_ylim()
max_mean_norm = float(np.nanmax(mean)) if np.isfinite(np.nanmax(mean)) else max_mean
if np.isfinite(max_mean_norm) and max_mean_norm > 0:
    upper = max(ymax, max_mean_norm)
    upper = min(upper, 2 * max_mean_norm)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_INTENSITY_NORM_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_NORM_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized intensity plot:", PLOT_INTENSITY_NORM_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Intensity, Mean Peak Normalized, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_intensity_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive intensity (mean peak normalized to 1)")
ax.set_title("Marker-positive intensity vs transverse distance (merged, normalized, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
ymin, ymax = ax.get_ylim()
max_mean_norm = float(np.nanmax(mean)) if np.isfinite(np.nanmax(mean)) else max_mean
if np.isfinite(max_mean_norm) and max_mean_norm > 0:
    upper = max(ymax, max_mean_norm)
    upper = min(upper, 2 * max_mean_norm)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_INTENSITY_NORM_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_NORM_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized intensity plot (CI):", PLOT_INTENSITY_NORM_CI_PATH.relative_to(ROOT).as_posix())


## Plot (All Intensity, Not Just Positive Mask)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_intensity_all_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_all_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_all_fraction"].to_numpy(dtype=float)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of all corrected intensity in bin")
ax.set_title("Marker intensity vs transverse distance (merged, all mask pixels)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
if np.isfinite(max_mean) and max_mean > 0:
    ymin, ymax = ax.get_ylim()
    upper = max(ymax, max_mean)
    upper = min(upper, 2 * max_mean)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_INTENSITY_ALL_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_ALL_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote all-intensity profile plot:", PLOT_INTENSITY_ALL_PATH.relative_to(ROOT).as_posix())


## Plot (All Intensity, Not Just Positive Mask, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_intensity_all_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_all_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_all_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of all corrected intensity in bin")
ax.set_title("Marker intensity vs transverse distance (merged, all mask pixels, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
if np.isfinite(max_mean) and max_mean > 0:
    ymin, ymax = ax.get_ylim()
    upper = max(ymax, max_mean)
    upper = min(upper, 2 * max_mean)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_INTENSITY_ALL_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_ALL_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote all-intensity profile plot (CI):", PLOT_INTENSITY_ALL_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Soft-threshold Intensity, Raw)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_soft_intensity_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["soft_intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_soft_intensity_fraction"].to_numpy(dtype=float)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of soft-threshold intensity in bin")
ax.set_title("Marker intensity vs transverse distance (merged, soft threshold)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
if np.isfinite(max_mean) and max_mean > 0:
    ymin, ymax = ax.get_ylim()
    upper = max(ymax, max_mean)
    upper = min(upper, 2 * max_mean)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_SOFT_RAW_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_SOFT_RAW_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote soft-threshold profile plot:", PLOT_SOFT_RAW_PATH.relative_to(ROOT).as_posix())


## Plot (Soft-threshold Intensity, Raw, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_soft_intensity_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["soft_intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_soft_intensity_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of soft-threshold intensity in bin")
ax.set_title("Marker intensity vs transverse distance (merged, soft threshold, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
if np.isfinite(max_mean) and max_mean > 0:
    ymin, ymax = ax.get_ylim()
    upper = max(ymax, max_mean)
    upper = min(upper, 2 * max_mean)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_SOFT_RAW_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_SOFT_RAW_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote soft-threshold profile plot (CI):", PLOT_SOFT_RAW_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Soft-threshold Intensity, Mean Peak Normalized)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["soft_intensity_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_soft_intensity_fraction"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Soft-threshold intensity (mean peak normalized to 1)")
ax.set_title("Marker intensity vs transverse distance (merged, soft threshold, normalized)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
ymin, ymax = ax.get_ylim()
max_mean_norm = float(np.nanmax(mean)) if np.isfinite(np.nanmax(mean)) else max_mean
if np.isfinite(max_mean_norm) and max_mean_norm > 0:
    upper = max(ymax, max_mean_norm)
    upper = min(upper, 2 * max_mean_norm)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_SOFT_NORM_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_SOFT_NORM_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized soft-threshold plot:", PLOT_SOFT_NORM_PATH.relative_to(ROOT).as_posix())


## Plot (Soft-threshold Intensity, Mean Peak Normalized, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = interp_df.loc[interp_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["soft_intensity_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_soft_intensity_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Soft-threshold intensity (mean peak normalized to 1)")
ax.set_title("Marker intensity vs transverse distance (merged, soft threshold, normalized, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)
ymin, ymax = ax.get_ylim()
max_mean_norm = float(np.nanmax(mean)) if np.isfinite(np.nanmax(mean)) else max_mean
if np.isfinite(max_mean_norm) and max_mean_norm > 0:
    upper = max(ymax, max_mean_norm)
    upper = min(upper, 2 * max_mean_norm)
    ax.set_ylim(0, upper)

fig.savefig(PLOT_SOFT_NORM_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_SOFT_NORM_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized soft-threshold plot (CI):", PLOT_SOFT_NORM_CI_PATH.relative_to(ROOT).as_posix())
